# Menon × Siletti × ASAP scVI+scANVI Integration Explorer

Interactive exploration of outputs from `05_integrate_menon_siletti_asap.py` (without Kamath data)

- **Section 1** — UMAP explorer: colour dots by any obs column. Load cells identified as DA neurons in the 4-source run. Plots show that they are ALSO grouped here in the 3-source run

In [ ]:
import os
import re
import sys
import glob
import functools
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import warnings

pio.renderers.default = 'plotly_mimetype'
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
repo_name = "spatial-sc-rnaseq-integration-wf"  # needed for get_repo_root to work in both unpromoted and live repo notebooks
sys.path.insert(0, str(Path.cwd().parent / "utils"))  # repo utils/
from notebook_utils import get_repo_root # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)

# ------------
# Folder name and string in h5ad with results from integrate_*.py
#
# Use downsampling_strategy = "asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers" to load the H5AD
# from the run with all 4-sources and Kamath's DA markers included
# downsampling_strategy = "asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers" 
#
# Use downsampling_strategy = "asap96k_menon99k_siletti100k_list_asap96k_menon99k_siletti100k_hvg_minus_da_markers_gene_list_no_kamath_no_kamath"
# to load the H5AD from the run with 3-sources (excluding Kamath's data) and DA markers also excluded
downsampling_strategy = "asap96k_menon99k_siletti100k_list_asap96k_menon99k_siletti100k_hvg_minus_da_markers_gene_list_no_kamath_no_kamath"
# ------------

h5ad_name = 'integrated_' + downsampling_strategy + '.h5ad'
OUTDIR_ROOT = repo_root / "reference_building" / "da_neurons_identification_analysis" / "output"
OUTPUT_DIR = str(OUTDIR_ROOT / downsampling_strategy)
REFERENCE_DATA_DIR = str(repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data")  # Intermediate H5ADs used as inputs for 04_integrate_kamath_menon_siletti_asap.py

## NOTE: Siletti datased labels provided by the Allen Institute (i.e including DA VGLUT2, not the ones at CELLxGENE)
REFERENCE_DATA_DIR_PATH = repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data"
REFERENCE_DATA_DIR = str(REFERENCE_DATA_DIR_PATH)
ALLEN_CELL_METADATA_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-10Xv3" / "20241115" / "cell_metadata.csv.gz")
ALLEN_MEMBERSHIP_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-taxonomy" / "20240330" / "cluster_to_cluster_annotation_membership.csv.gz")

### Cell types and marker genes

In [ ]:
# ------------- KAMATH ANNOTATIONS -------------
# DA subtype vulnerability labels (Kamath et al., Nat Neurosci 2022, Figure 3A & 3B)
# Vulnerability statistical values were extracted from the main text.
_DA_PD_SUFFIXES = {
    'SOX6_AGTR1':        'SOX6_AGTR1_vulnerable',  # significantly vulnerable (OR = −0.82, FDR-adjusted P < 0.05)
    'SOX6_DDT':          'SOX6_DDT',
    'SOX6_PART1':        'SOX6_PART1_down_ns',     # on the vulnerable side, but not cited as significant (no full stats tables were found)
    'SOX6_GFRA2':        'SOX6_GFRA2_up_ns',       # on the resilent side, but not cited as significant (no full stats tables were found)
    'CALB1_CALCR':       'CALB1_CALCR',
    'CALB1_RBP4':        'CALB1_RBP4',
    'CALB1_PPP1R17':     'CALB1_PPP1R17',
    'CALB1_CRYM_CCDC68': 'CALB1_CRYM_CCDC68',
    'CALB1_GEM':         'CALB1_GEM_resistant',    # significantly resistant (OR = 1.46, FDR-adjusted P < 0.05)
    'CALB1_TRHR':        'CALB1_TRHR_resistant',   # significantly resistant (OR = 0.84, FDR-adjusted P < 0.05)
}

# ------------- EACH DATASET CELL TYPE LABELS -------------
# See "`obs` column reference for details"
_NATIVE_COLS_BY_SOURCE = {
    'kamath': ('cell_type_scanvi',       # Kamath cell type used by 04_integrate_kamath_menon_siletti_asap.py to train scANVI for label propagation 
               'da_subtype'),            # DA-only subset of cell_type_scanvi
    'menon': ('menon_cell_type',         # Menon full cell type description
              'menon_author_cell_type'), # Abbreviated description. Maps 1:1 with menon_cell_type
    'siletti': (
        'siletti_cxg_cell_type', 'siletti_cxg_author_cell_type', # Siletti, originally deposited in CELLxGENE
        'siletti_allen_da_vglut2_label', 'siletti_allen_cluster', 'siletti_allen_cell_type', # Re-annotated by the Allen Institute (incl. "DA VGLUT2")
    ),
    'asap_cohort': ('pmdbs_sc_rnaseq.C_scANVI',   # scANVI-propagated labels from the ASAP pmdbs_sc_rnaseq pipeline (not from 04_integrate_kamath_menon_siletti_asap.py)
                    'pmdbs_sc_rnaseq.cell_type'),  # MMC / Allen Brain Atlas labels from the ASAP pmdbs_sc_rnaseq pipeline (not from 04_integrate_kamath_menon_siletti_asap.py)
}

### Load h5ad from integration

In [ ]:
# ── Load integrated AnnData (used by all sections below) ──────────────────
print('Loading integrated.h5ad ... see `reference_building/README.md`  for description of `obs`')
adata = ad.read_h5ad(os.path.join(OUTPUT_DIR, h5ad_name))
print(f'  {adata.n_obs:,} cells · {adata.n_vars:,} genes')
print(f'  Sources: {dict(adata.obs["source"].value_counts())}')
print(f'  obs columns: {adata.obs.columns.tolist()}')

In [ ]:
# <dataset>_DA obs columns: per-cell 'DA'/'non-DA' call using each source's
# OWN independent ground truth (not scANVI's cell_type_pred) -- for QC/
# benchmark use, not training. Each column applies only its own source's
# criterion; cells from other sources (and non-DA cells within the source)
# fall through to 'non-DA'.
#   menon_DA   -- menon_author_cell_type == 'Dopaminergic'
#   siletti_DA -- siletti_allen_da_vglut2 (boolean) == True. Uses the raw
#     boolean rather than the derived 'siletti_allen_da_vglut2_label' string
#     column, since that one isn't built until the next section below --
#     same underlying condition either way (True <-> 'DA VGLUT2').
#   kamath_DA  -- cell_type_scanvi is one of the 10 Kamath DA subtypes
#     (_DA_PD_SUFFIXES keys, e.g. 'SOX6_AGTR1', 'CALB1_GEM').
adata.obs['menon_DA'] = np.where(adata.obs['menon_author_cell_type'] == 'Dopaminergic', 'DA', 'non-DA')
adata.obs['siletti_DA'] = np.where(adata.obs['siletti_allen_da_vglut2'].astype(bool), 'DA', 'non-DA')
adata.obs['kamath_DA'] = np.where(adata.obs['cell_type_scanvi'].isin(_DA_PD_SUFFIXES), 'DA', 'non-DA')

for col in ('menon_DA', 'siletti_DA', 'kamath_DA'):
    print(adata.obs[col].value_counts().to_string())
    print()

In [ ]:
# Shared "Colour by" / crosstab-axis options per source.
_NATIVE_COLS_BY_SOURCE = {
    'kamath': ('cell_type_scanvi', 'da_subtype'),
    'menon': ('menon_cell_type', 'menon_author_cell_type'),
    'siletti': (
        'siletti_cxg_cell_type', 'siletti_cxg_author_cell_type', 'siletti_allen_da_vglut2_label',
        'siletti_allen_cluster', 'siletti_allen_cell_type',
    ),
    'asap_cohort': ('pmdbs_sc_rnaseq.C_scANVI', 'pmdbs_sc_rnaseq.cell_type'),
}


# subcluster -> cluster is a verified stable many-to-one mapping in Allen's taxonomy.
def _load_siletti_subcluster_to_cluster():
    membership = pd.read_csv(ALLEN_MEMBERSHIP_PATH, keep_default_na=False)
    sub = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'subcluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'subcluster_name'})
    clu = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'cluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'cluster_name'})
    merged = sub.merge(clu, on='cluster_alias').drop_duplicates('subcluster_name')
    return merged.set_index('subcluster_name')['cluster_name']


# Materialize Siletti's derived label columns once (NaN for non-Siletti cells).
if 'siletti_allen_da_vglut2' in adata.obs.columns:
    _siletti_mask = adata.obs['source'] == 'siletti'
    adata.obs['siletti_allen_da_vglut2_label'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2_label'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2']
        .map({True: 'DA VGLUT2 (Allen)', False: 'non-DA'})
    )
    adata.obs['siletti_allen_cluster'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_cluster'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_subcluster']
        .map(_load_siletti_subcluster_to_cluster())
    )
    # Coarse cell type -- abbreviated prefix of siletti_allen_cluster (e.g. 'Astro_60' -> 'Astro').
    adata.obs['siletti_allen_cell_type'] = (
        adata.obs['siletti_allen_cluster'].str.replace(r'_\d+$', '', regex=True)
    )
    print(
        f"  Derived siletti_allen_da_vglut2_label, siletti_allen_cluster, "
        f"siletti_allen_cell_type "
        f"({int(adata.obs['siletti_allen_cluster'].notna().sum()):,} Siletti cells)"
    )

In [ ]:
# Gene symbol -> Ensembl ID mapping, via Menon's 'feature_name' var column
# (covers every gene in the merged h5ad, since it's an inner join subset).
_menon_cache_paths = glob.glob(f'{REFERENCE_DATA_DIR}/menon/intermediate/*_sn_only*.h5ad')
if _menon_cache_paths:
    _menon_var = sc.read_h5ad(_menon_cache_paths[0], backed='r').var
    adata.var['gene_symbol'] = _menon_var['feature_name'].reindex(adata.var_names)
    print(f"  Gene symbols resolved for {adata.var['gene_symbol'].notna().sum():,}/{adata.n_vars:,} "
          f"genes (via {_menon_cache_paths[0]})")
else:
    adata.var['gene_symbol'] = np.nan
    print("  Warning: no Menon SN-only cache found -- gene_symbol lookup unavailable.")

## 1. UMAP Explorer

Scatter plot of the scANVI-latent UMAP (`obsm['X_umap']`). Use the **Colour by** dropdown to switch between any obs column. Key analysis columns are listed first.

- Categorical / low-cardinality integer columns → discrete colour palette
- Continuous numeric columns → Viridis scale
- Cells with `NaN` in the selected column appear in light grey

Hover over a point to see its `source`, `cell_type_pred`, `cell_type_scanvi`, `da_subtype`, Menon native labels, and ASAP PMDBS sc-RNA-seq native labels.

In [ ]:
if 'X_umap' not in adata.obsm:
    raise KeyError(
        'X_umap not found in adata.obsm. '
        'Re-run integrate_kamath_menon_asap.py with --compute-umap.'
    )

umap_df = pd.DataFrame(
    adata.obsm['X_umap'],
    columns=['UMAP_1', 'UMAP_2'],
    index=adata.obs_names,
)
umap_df = pd.concat([umap_df, adata.obs], axis=1)

# Apply PD vulnerability suffixes for display (h5ad labels are unchanged)
for col in ('cell_type_pred', 'cell_type_scanvi', 'da_subtype'):
    if col in umap_df.columns:
        umap_df[col] = umap_df[col].replace(_DA_PD_SUFFIXES)

In [ ]:
from notebook_utils import plot_umaps_categ

# Binds this notebook's native-column/PD-vulnerability display config so
# existing call sites below (_plot_umaps_categ(adata=...)) don't need to change.
_plot_umaps_categ = functools.partial(
    plot_umaps_categ,
    native_cols_by_source=_NATIVE_COLS_BY_SOURCE,
    display_label_remap=_DA_PD_SUFFIXES,
)

### Colour UMAP plots by cell types

In [ ]:
# Show UMAP and save lasso-selected barcode ID's to `selected_barcodes`
selected_barcodes = _plot_umaps_categ(adata=adata)

### Colour barcodes selected in the 4-source notebook

In [ ]:
selected_barcodes_4_source = OUTDIR_ROOT / 'asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers' / 'selected_barcodes_source.txt'

with open(selected_barcodes_4_source) as f:
    selected_4_source_da_barcodes = set(line.strip() for line in f if line.strip())

adata.obs['selected_4_source_da'] = adata.obs_names.isin(selected_4_source_da_barcodes)

n_matched = adata.obs['selected_4_source_da'].sum()
print(f"{n_matched:,}/{len(selected_4_source_da_barcodes):,} selected barcodes matched in this H5AD "
      f"({adata.obs['selected_4_source_da'].sum():,}/{adata.n_obs:,} cells flagged True)")

In [ ]:
_plot_umaps_categ(adata=adata)

In [ ]:
adata.obs['source'].value_counts()

In [ ]:
adata_menon = adata[adata.obs['source'] == "menon"]
adata_siletti = adata[adata.obs['source'] == "siletti"]
adata_asap = adata[adata.obs['source'] == "asap_cohort"]

In [ ]:
_plot_umaps_categ(adata=adata_menon)

In [ ]:
_plot_umaps_categ(adata=adata_asap)